# Kafka Learning Notes — Part 1: Topics, Partitions, and Offsets

## 1. What Is Kafka?

Apache Kafka is a **distributed event-streaming platform** used to publish, store, and consume streams of messages or events.

Its main components include:

- **Producers:** Applications that publish events to Kafka.
- **Topics:** Named streams that organize related events.
- **Consumers:** Applications that read and process events.
- **Brokers:** Servers that store events and handle producer and consumer requests.

## 2. Topics and Partitions

A Kafka topic groups related events, such as orders, payments, or application logs.

Each topic contains **one or more partitions**, such as partition 0, partition 1, partition 2, and partition 3. Each partition is an ordered log of records, with new records appended to its end.

Kafka preserves the order of records **within a partition**, but it does not guarantee a single order across all partitions in a topic.

A topic is better understood as a **partitioned event log** than as a database without a schema. Kafka stores serialized keys and values, while applications define their data formats and may use schemas.

## 3. Message Keys and Partition Assignment

A producer can include a **key** with each message, such as a customer ID or an order ID.

### Messages with a Key

With the standard key-based partitioning strategy, messages with the same key go to the same partition, provided the partition count and partitioning strategy remain unchanged.

For example, messages with the key `customer-123` are routed to the same partition, keeping that customer's events together.

Different keys can also map to the same partition.

### Messages Without a Key

When the key is `null`, partition assignment depends on the producer client and configuration.

Modern Kafka Java producers use **sticky partitioning behavior** by default for unkeyed records. They keep records together in a partition to improve batching before selecting another partition.

**Round-robin partitioning** is an available strategy, but it is not always the default for messages without a key.

## 4. Connecting Multiple Producers and Consumers

Kafka reduces the need for direct connections between every producer and consumer.

For example, producers A and B can publish events to a topic, while notification, analytics, and auditing applications consume those events independently.

Producers do not need to know which applications consume their events.

### Consumer Groups

- **Different consumer groups** can independently read the same events.
- **Consumers within a standard consumer group** share the work by dividing partitions among themselves.

Kafka supports event-driven communication, while applications implement any required workflow orchestration.

## 5. Horizontal Scaling

Kafka supports **horizontal scaling** by distributing partitions across brokers.

Consumers can also process different partitions in parallel. Within a standard consumer group, each partition is assigned to one consumer at a time.

For example, a topic with four partitions supports up to four consumers actively processing those partitions within the same group. Additional consumers in that group remain idle until a partition becomes available.

## 6. Offsets

An **offset** identifies a record's position within a partition.

Offsets are specific to each partition. For example, partition 0 and partition 1 can both contain a record at offset 5; those offsets identify different records.

Consumers use offsets to track their progress and can return to earlier offsets to replay records that Kafka still retains.

## 7. Immutability and Data Retention

Kafka records are **immutable**: once written, an existing record cannot be edited in place. A producer represents a change by publishing a new record.

However, immutability does not mean that records are stored forever:

- **Retention policies** can remove older records.
- **Log compaction** can remove older records for a key while retaining its latest value.
- Removed records leave gaps in the offsets. Remaining offsets are not renumbered, and removed offsets are not reused within that partition's lifetime.

Reading a record does not automatically delete it.

Kafka allows consumers to read and replay retained records by offset, but it does not directly provide ordinary database-style queries over a topic's records.